# Notebook 1: PATSTAT High-Recall Baseline Extraction
Extracts the broad candidate corpus of drone-related patent applications from PATSTAT `PROD` using boolean combinations of CPC/IPC classes and domain keywords, runs initial quality diagnostics, and saves the raw baseline to Parquet.

In [ ]:
import os
from datetime import date

import pandas as pd
from sqlalchemy import func, or_, and_, not_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,               # applications
    TLS202_APPLN_TITLE,         # titles
    TLS203_APPLN_ABSTR,         # abstracts
    TLS206_PERSON,              # persons (applicants and inventors)
    TLS207_PERS_APPLN,          # person <-> application links
    TLS209_APPLN_IPC,           # IPC classification
    TLS224_APPLN_CPC,           # CPC classification
    TLS231_INPADOC_LEGAL_EVENT, # legal events
)

In [ ]:
# Connect to PATSTAT PROD environment
patstat = PatstatClient(env="PROD")
db = patstat.orm()

In [ ]:
# Multi-tier taxonomy and exclusion lists

# Primary Anchor Keywords (High Precision: Directly define flying UAV platforms)
DRONE_ANCHOR_KEYWORDS = [
    "%unmanned aerial vehicle%",
    "%unmanned aircraft%",
    "%remotely piloted aircraft%",
    "%RPAS%",
    "%UAV%",
]

# Secondary Functional Keywords (Dual-Use: Require co-occurrence with Core CPC/IPC or Anchor Keywords)
DRONE_SECONDARY_KEYWORDS = [
    "%drone%",
    "%quadcopter%",
    "%multirotor%",
    "%hexacopter%",
]

# Exclusion Keywords (Filter out non-flying autonomous/robotic domains)
DRONE_EXCLUSION_KEYWORDS = [
    "%toy%",
    "%underwater%",
    "%submersible%",
    "%marine%",
]


DRONE_CORE_CPC_CODES = [
    "B64U%",        # Primary CPC class dedicated to UAVs
    "B64C39/02%",   # Legacy CPC class for remote-controlled aircraft
]

DRONE_SECONDARY_CPC_CODES = [
    "G08G5/00%",    # Aeronautical traffic control & UTM
    "G05D1/00%",    # Control of position / course / flight control
]

DRONE_CORE_IPC_CODES = [
    "B64U%",
    "B64C39/02%",
]

DRONE_SECONDARY_IPC_CODES = [
    "G08G5/00%",
    "G05D1/00%",
]

In [ ]:
# Build Atomic Filter Conditions
core_cpc_f = [TLS224_APPLN_CPC.cpc_class_symbol.like(c) for c in DRONE_CORE_CPC_CODES]
core_ipc_f = [TLS209_APPLN_IPC.ipc_class_symbol.like(c) for c in DRONE_CORE_IPC_CODES]

sec_cpc_f = [TLS224_APPLN_CPC.cpc_class_symbol.like(c) for c in DRONE_SECONDARY_CPC_CODES]
sec_ipc_f = [TLS209_APPLN_IPC.ipc_class_symbol.like(c) for c in DRONE_SECONDARY_IPC_CODES]

title_anchor_f = [TLS202_APPLN_TITLE.appln_title.ilike(k) for k in DRONE_ANCHOR_KEYWORDS]
abstr_anchor_f = [TLS203_APPLN_ABSTR.appln_abstract.ilike(k) for k in DRONE_ANCHOR_KEYWORDS]

title_sec_f = [TLS202_APPLN_TITLE.appln_title.ilike(k) for k in DRONE_SECONDARY_KEYWORDS]
abstr_sec_f = [TLS203_APPLN_ABSTR.appln_abstract.ilike(k) for k in DRONE_SECONDARY_KEYWORDS]

title_excl_f = [TLS202_APPLN_TITLE.appln_title.ilike(k) for k in DRONE_EXCLUSION_KEYWORDS]
abstr_excl_f = [TLS203_APPLN_ABSTR.appln_abstract.ilike(k) for k in DRONE_EXCLUSION_KEYWORDS]

In [ ]:
# Restructured Boolean Logic Clauses
standalone_core = or_(
    *core_cpc_f,
    *core_ipc_f,
    *title_anchor_f,
    *abstr_anchor_f
)

# Secondary codes MUST co-occur with explicit Anchor terms
secondary_code_match = and_(
    or_(*sec_cpc_f, *sec_ipc_f),
    or_(*title_anchor_f, *abstr_anchor_f)
)

# Secondary keywords MUST co-occur with Core Codes
secondary_kw_match = and_(
    or_(*title_sec_f, *abstr_sec_f),
    or_(*core_cpc_f, *core_ipc_f)
)

# Master Conditions
inclusion_clause = or_(
    standalone_core,
    secondary_code_match,
    secondary_kw_match
)

exclusion_clause = not_(or_(*title_excl_f, *abstr_excl_f))

In [ ]:
MIN_FILING_DATE = date(2000, 1, 1)
MAX_FILING_DATE = date(2023, 1, 1)

query = (
    db.query(
        TLS201_APPLN.appln_id,
        TLS201_APPLN.docdb_family_id,
        TLS201_APPLN.appln_auth,
        TLS201_APPLN.appln_nr,
        TLS201_APPLN.appln_filing_date,
        TLS201_APPLN.appln_kind,
        TLS202_APPLN_TITLE.appln_title,
        TLS203_APPLN_ABSTR.appln_abstract,
        func.string_agg(func.distinct(TLS206_PERSON.doc_std_name), ', ').label("applicants"),
        func.string_agg(func.distinct(TLS224_APPLN_CPC.cpc_class_symbol), ', ').label("cpc_codes"),
        func.string_agg(func.distinct(TLS209_APPLN_IPC.ipc_class_symbol), ', ').label("ipc_codes"),
    )
    .join(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
    .outerjoin(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
    .outerjoin(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
    .join(TLS207_PERS_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id)
    .join(TLS206_PERSON, TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id)
    .filter(
        and_(
            inclusion_clause,
            exclusion_clause,
            TLS201_APPLN.appln_filing_date >= MIN_FILING_DATE,
            TLS201_APPLN.appln_filing_date < MAX_FILING_DATE
        )
    )
    .group_by(
        TLS201_APPLN.appln_id,
        TLS201_APPLN.docdb_family_id,
        TLS201_APPLN.appln_auth,
        TLS201_APPLN.appln_nr,
        TLS201_APPLN.appln_filing_date,
        TLS201_APPLN.appln_kind,
        TLS202_APPLN_TITLE.appln_title,
        TLS203_APPLN_ABSTR.appln_abstract
    )
)

### Execute Query on PATSTAT PROD
**Warning:** Running `query.all()` queries the full production database and takes several minutes. Only run when regenerating the raw baseline.

In [ ]:
results = query.all()
df = pd.DataFrame(results)
print(f"Total matching patent applications retrieved: {len(df):,}")

In [ ]:
# Baseline Data Diagnostics
print("Abstract missingness:", round(df['appln_abstract'].isna().mean(), 4))
print("Title missingness:", round(df['appln_title'].isna().mean(), 4))
print("Total Applications (appln_id):", df['appln_id'].nunique())
print("Total Patent Families (docdb_family_id):", df['docdb_family_id'].nunique())

df['filing_year'] = pd.to_datetime(df['appln_filing_date'], errors='coerce').dt.year
print("\nFiling year distribution:")
print(df['filing_year'].value_counts().sort_index())
print("\nPlaceholder/Invalid date count:", df['filing_year'].isna().sum())

df.head()

In [ ]:
# Save baseline corpus to Parquet
baseline_path = "flying_drones_baseline_prod.parquet"
df.to_parquet(baseline_path, index=False)

size_mb = os.path.getsize(baseline_path) / (1024 * 1024)
print(f"Successfully saved '{baseline_path}': {len(df):,} rows ({size_mb:.2f} MB)")